In [1]:
using CairoMakie
include("../src-fig/plotting.jl")

include("../src-fig/figures/frontogenesis.jl")
ts_grow = [50, 72.5, 70, 67.5, 65, 65, 57.5];
ts_max = [80, 112.5, 127.5, 127.5, 127.5, 132.5, 132.5];
ts_decay = [100, 162.5, 175, 177.5, 192.5, 197.5, 200];
ts_start = [25, 25, 25, 25, 25, 25, 25];

[ Info: Oceananigans will use 4 threads


# Surface energy spectra

In [2]:
for (ts, name) in zip([ts_grow, ts_max, ts_decay, ts_start], ["grow", "max", "decay", "start"])
    fig = Figure(; size=(800, 400), fontsize=18)
    ax = Axis(fig[1, 1]; 
        xscale = log10, 
        yscale = log10, 
        title = L"Surface energy spectrum at $t=t_\text{%$name}$",
        xreversed = true, 
        xlabel = L"\lambda / \text{m}",
        ylabel = L"\text{Energy density} / \text{m}^2\,\text{s}^{-2}"
    )
    colors = Makie.wong_colors()
    ns = 1:7
    lns = map(ns) do n
        run_id = ensemble.cooling_set.filenames[n]
        SURFACE = joinpath(scratchpath, run_id, "SURFACE.jld2")
        INS = joinpath(scratchpath, run_id, "INS.jld2")
        sp = simulation_parameters(SURFACE)
        
        iterations, times = iterations_times(SURFACE)
        i = argmin(abs.(ts[n] / sp.f .- times))
        u = get_field(SURFACE, "u", iterations[i])
        v = get_field(SURFACE, "v", iterations[i])
        (λs, uu) = modedecomposition(u, u, sp.Lx / sp.Nx)
        (λs, vv) = modedecomposition(v, v, sp.Lx / sp.Nx)
        ln = lines!(ax, λs, (uu .+ vv) ./ 2; color=colors[n])
        ln
    end
    lines!(ax, [1e4, 1e5], k->3e8(k/1e5)^(5/3); color=(:black, 0.8))
    lines!(ax, [1e4, 4e4], k->5e8(k/1e5)^(3); color=(:black, 0.8))
    Legend(fig[1, 2], lns, map(run_label, ensemble.cooling_set.filenames[ns]), legend_title)
    save("figures/energy-spectrum-$name.png", fig; px_per_unit=2)
    fig
end

# Vorticity

In [3]:
for (ts, name) in zip([ts_grow, ts_max, ts_decay, ts_start], ["grow", "max", "decay", "start"])
    fig = Figure(; size=(800, 400), fontsize=18)
    ax = Axis(fig[1, 1]; 
        xscale = log10, 
        yscale = log10, 
        title = L"Surface enstrophy spectrum at $t=t_\text{%$name}$",
        xreversed = true, 
        xlabel = L"\lambda / \text{m}",
        ylabel = L"\text{Enstrophy} / \text{s}^{-2}"
    )
    colors = Makie.wong_colors()
    ns = 1:7
    lns = map(ns) do n
        run_id = ensemble.cooling_set.filenames[n]
        STRAINTENSOR = joinpath(scratchpath, run_id, "STRAINTENSOR.jld2")
        iterations, times = iterations_times(STRAINTENSOR)
        sp = simulation_parameters(STRAINTENSOR)
        i = argmin(abs.(ts[n] / sp.f .- times))
        ζ = get_field(STRAINTENSOR, "ζ", iterations[i])
        (λs, ζζ) = modedecomposition(ζ, ζ, sp.Lx / sp.Nx)
        lines!(ax, λs, ζζ ./ 2; color=colors[n])
    end
    lines!(ax, [1e4, 7e4], k->5e1(k / 1e5)^0; color=(:black, 0.8))
    Legend(fig[1, 2], lns, map(run_label, ensemble.cooling_set.filenames[ns]), legend_title)
    save("figures/enstrophy-spectrum-$name.png", fig; px_per_unit=2)
    fig
end

# Surface buoyancy

In [4]:
for (ts, name) in zip([ts_grow, ts_max, ts_decay, ts_start], ["grow", "max", "decay", "start"])
    fig = Figure(; size=(800, 400), fontsize=18)
    ax = Axis(fig[1, 1]; 
        xscale = log10, 
        yscale = log10, 
        xreversed = true, 
        title = L"Surface buoyancy spectrum at $t=t_\text{%$name}$",
        xlabel = L"\lambda / \text{m}",
        ylabel = L"\text{Buoyancy}^2 / \text{m}^2\,\text{s}^{-4}"
    )
    colors = Makie.wong_colors()
    ns = 1:7
    lns = map(ns) do n
        run_id = ensemble.cooling_set.filenames[n]
        SURFACE = joinpath(scratchpath, run_id, "SURFACE.jld2")
        INS = joinpath(scratchpath, run_id, "INS.jld2")
        sp = simulation_parameters(SURFACE)
        
        iterations, times = iterations_times(SURFACE)
        i = argmin(abs.(ts[n] / sp.f .- times))
        b = get_field(SURFACE, "b", iterations[i])
        (λs, bb) = modedecomposition(b, b, sp.Lx / sp.Nx)
        ln = lines!(ax, λs, bb ./ 2; color=colors[n])
        #=
        iterations, times = iterations_times(INS)
        i = argmin(abs.(ts[n] / sp.f .- times))
        b = mean(get_field(INS, "b", iterations[i]); dims=3)[:, :, 1]
        (λs, bb) = modedecomposition(b, b, sp.Lx / sp.Nx)
        lines!(ax, λs, bb ./ λs.^4 ./ 2; color=colors[n], linestyle=:dash)
        =#
        ln
    end
    lines!(ax, [1e4, 1e5], k->1.7e2(k/1e5)^(2); color=(:black, 0.8))
    lines!(ax, [1e4, 1e5], k->1.7e2(k/1e5)^(4); color=(:black, 0.8))
    Legend(fig[1, 2], lns, map(run_label, ensemble.cooling_set.filenames[ns]), legend_title)
    save("figures/buoyancy-spectrum-$name.png", fig; px_per_unit=2)
    fig
end

# Passive tracer

In [5]:
for (ts, name) in zip([ts_grow, ts_max, ts_decay, ts_start], ["grow", "max", "decay", "start"])
    fig = Figure(; size=(800, 400), fontsize=18)
    ax = Axis(fig[1, 1]; 
        xscale = log10, 
        yscale = log10, 
        xreversed = true, 
        title = L"Surface tracer spectrum at $t=t_\text{%$name}$",
        xlabel = L"\lambda / \text{m}",
        ylabel = L"\text{Passive tracer}"
    )
    colors = Makie.wong_colors()
    ns = 1:7
    lns = map(ns) do n
        run_id = ensemble.cooling_set.filenames[n]
        SURFACE = joinpath(scratchpath, run_id, "SURFACE.jld2")
        INS = joinpath(scratchpath, run_id, "INS.jld2")
        sp = simulation_parameters(SURFACE)
        
        iterations, times = iterations_times(SURFACE)
        i = argmin(abs.(ts[n] / sp.f .- times))
        b = get_field(SURFACE, "c", iterations[i])
        (λs, bb) = modedecomposition(b, b, sp.Lx / sp.Nx)
        ln = lines!(ax, λs, bb ./ 2; color=colors[n])
        #=
        iterations, times = iterations_times(INS)
        i = argmin(abs.(ts[n] .- times))
        b = mean(get_field(INS, "c", iterations[i]); dims=3)[:, :, 1]
        (λs, bb) = modedecomposition(b, b, sp.Lx / sp.Nx)
        lines!(ax, λs, bb ./ 2; color=colors[n], linestyle=:dash)
        =#
        ln
    end
    lines!(ax, [1e4, 1e5], k->1.5e4(k/1e6)^(2); color=(:black, 0.5))
    Legend(fig[1, 2], lns, map(run_label, ensemble.cooling_set.filenames[ns]), legend_title)
    save("figures/tracer-spectrum-$name.png", fig; px_per_unit=2)
    fig
end

# Frontogenesis

In [7]:
fig = frontogenesisspectra(ensemble.cooling_set.filenames[3:7], ts_grow[3:7], Makie.wong_colors()[3:7])
save("figures/frontogenesis-spectra-grow.png", fig; px_per_unit=2)

fig = frontogenesisspectra(ensemble.cooling_set.filenames[3:7], ts_max[3:7], Makie.wong_colors()[3:7])
save("figures/frontogenesis-spectra-max.png", fig; px_per_unit=2)

fig = frontogenesisspectra(ensemble.cooling_set.filenames[3:7], ts_decay[3:7], Makie.wong_colors()[3:7])
save("figures/frontogenesis-spectra-decay.png", fig; px_per_unit=2)

fig = frontogenesisspectra(ensemble.cooling_set.filenames[3:7], ts_start[3:7], Makie.wong_colors()[3:7])
save("figures/frontogenesis-spectra-start.png", fig; px_per_unit=2)

4: 75.0
4: 75.0
14: 65.0
4: 75.0
3: 50.0
6: 125.0
6: 125.0
22: 125.0
6: 125.0
6: 125.0
8: 175.0
8: 175.0
25: 200.0
9: 200.0
9: 200.0
2: 25.0
2: 25.0
6: 25.0
2: 25.0
2: 25.0
